# Run ERM (SGNM equivariant model) on a GPU

Runtime → **Change runtime type → GPU (T4 is fine)**, then **Runtime → Run all**.

Produces `results/erm_profiles.csv` and downloads it. Back on your Mac:
```
# drop the file into results/, then:
python3 scripts/ingest_erm.py
```
which merges ERM into `scores.csv` and regenerates all plots + `results.html`.

In [ ]:
# 1. System + Python deps
!apt-get -qq install -y gperf >/dev/null   # ciffy C extension needs gperf
!pip -q install numpy pandas scipy h5py tqdm   # sgnm import-time + runtime deps

# dlu (pure python; dist name is 'dlu-torch', imports as 'dlu')
!pip -q install git+https://github.com/hmblair/dlu.git

# ciffy: upstream pyproject has a DUPLICATE fallback_version that breaks the
# build (tomllib 'Cannot overwrite a value'). Clone, patch, then install.
!rm -rf /content/ciffy && git clone -q https://github.com/hmblair/ciffy.git /content/ciffy
import pathlib
_pp = pathlib.Path('/content/ciffy/pyproject.toml')
_pp.write_text(_pp.read_text().replace('fallback_version = "0.0.0"\n', '', 1))
!pip -q install /content/ciffy

# flash-eq: prebuilt CUDA wheels; fall back to building from source (Colab has nvcc)
!pip -q install flash-eq --extra-index-url https://hmblair.github.io/flash-eq/ || \
  pip -q install git+https://github.com/hmblair/flash-eq.git

# sgnm: its pyproject pins 'dlu' (the dist is 'dlu-torch'), so install --no-deps
!pip -q install --no-deps git+https://github.com/hmblair/sgnm.git

import torch, ciffy, dlu, sgnm
from sgnm import EquivariantReactivityModel
print('imports OK | CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > GPU'

In [ ]:
# 2. Get this project's data + scripts, and the ERM checkpoint
%cd /content
!rm -rf rna-shape-decoys && git clone -q https://github.com/rhiju/rna-shape-decoys.git
%cd rna-shape-decoys
!curl -sL -o equivariant-checkpoint.pth \
  https://github.com/hmblair/sgnm/releases/download/v2.0.2/equivariant-checkpoint.pth
!ls -lh equivariant-checkpoint.pth   # should be ~1 MB (not a few bytes)

In [ ]:
# 3. Sanity check: ciffy must type atoms correctly (column-aligned converter)
import sys; sys.path.insert(0, 'scripts')
import ciffy, numpy as np, tempfile
from pdb_to_cif import pdb_to_cif
with tempfile.NamedTemporaryFile(suffix='.cif') as tf:
    pdb_to_cif('data/farfar2/Mol9_reference_UtoG_buildloop.pdb', tf.name)
    p = ciffy.load(tf.name, backend='numpy')
a = np.asarray(p.atoms)
print('code-0 atoms (must be 0):', int((a == 0).sum()), '/', len(a))
assert (a == 0).sum() == 0, 'atom typing failed — check pdb_to_cif'

In [ ]:
# 4. Run ERM on all structures (writes results/erm_profiles.csv)
!python3 scripts/run_erm.py equivariant-checkpoint.pth

In [ ]:
# 5. (optional GPU sanity check) re-run SGNM/GNM on GPU and compare to Mac CPU
!curl -sL -o gnm-checkpoint.pth \
  https://github.com/hmblair/sgnm/releases/download/v2.0.2/gnm-checkpoint.pth
!python3 scripts/03_shape_sgnm.py 2>/dev/null | tail -3 || echo 'skip if checkpoint path differs'

In [ ]:
# 6. Download results/erm_profiles.csv
from google.colab import files
files.download('results/erm_profiles.csv')